# 🤖 VisionTracker Remote Server — Google Colab Setup

This notebook sets up a self-hosted **Gemma 3 4B** vision-language model as a FastAPI server for VisionTracker object identification.

## Features
- 🆓 **Free tier compatible** — runs on Colab's free T4 GPU
- 🔒 **Private** — your images never leave your server
- ⚡ **Higher throughput** — no rate limits (you control the server)
- 🌐 **Accessible anywhere** — via ngrok tunnel

## Prerequisites
1. Create a free ngrok account at https://dashboard.ngrok.com/signup
2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. (Optional) Set it as a Colab secret: Click 🔑 in left sidebar → Add secret: `NGROK_AUTHTOKEN`

## Cell 1: Install Dependencies

Installs FastAPI, llama-cpp-python (GPU-accelerated), ngrok, and other requirements.

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic

# Install llama-cpp-python with CUDA support for Colab's T4 GPU
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python

print("✅ Dependencies installed!")

## Cell 2: Configure ngrok

Set up the ngrok authentication token to create secure tunnels.

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

# Try to get token from Colab secrets, fallback to manual input
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    print("✅ Loaded NGROK_AUTHTOKEN from Colab secrets")
except Exception:
    NGROK_TOKEN = input("Enter your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken): ").strip()

if not NGROK_TOKEN:
    raise ValueError("ngrok authtoken is required!")

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok configured!")

## Cell 3: Download Gemma 3 4B Model

Downloads the **Gemma 3 4B IT** model in GGUF format (Q4_K_M quantization).

- **Model size**: ~3.3GB (Q4_K_M quantized)
- **Original model**: google/gemma-3-4b-it
- **Quantized by**: bartowski/lmstudio-community
- **VRAM required**: ~4GB (fits in Colab T4's 16GB)

First time download takes ~2-3 minutes. The model is cached for future runs.

In [ ]:
import os

# Model configuration
MODEL_URL = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"
MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"
MODEL_PATH = f"/content/{MODEL_FILENAME}"

# Download if not exists
if not os.path.exists(MODEL_PATH):
    print(f"⬇️ Downloading {MODEL_FILENAME}...")
    print(f"   This is ~3.3GB, will take 2-3 minutes...")
    !wget -q --show-progress {MODEL_URL} -O {MODEL_PATH}
    print("✅ Download complete!")
else:
    print(f"✅ Model already exists at {MODEL_PATH}")

print(f"📦 Model size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

## Cell 4: Load Model & Warmup

Loads the model with llama-cpp-python and runs a warmup inference.

⚠️ **Note**: First inference is slow as CUDA kernels initialize. Warmup ensures subsequent requests are faster.

In [ ]:
from llama_cpp import Llama

print("🔄 Loading Gemma 3 4B model...")
print("   This may take 30-60 seconds...")

# Load model with GPU acceleration
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,          # Context window
    n_gpu_layers=-1,     # Use all layers on GPU (-1 = auto)
    verbose=False
)

print("✅ Model loaded!")

# Warmup inference
print("🔥 Running warmup inference...")
_ = llm("Say 'ready'", max_tokens=10)
print("✅ Warmup complete! Server is ready to accept connections.")

## Cell 5: Create FastAPI Server

Creates the FastAPI application with endpoints for object identification.

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import List, Optional
from PIL import Image
import io
import base64

app = FastAPI(title="VisionTracker Remote ID Server", version="1.0.0")

# Optional API key for security (set via Colab secrets: SERVER_API_KEY)
try:
    SERVER_API_KEY = userdata.get('SERVER_API_KEY')
except:
    SERVER_API_KEY = None

class IdentifyRequest(BaseModel):
    images: List[str]  # List of base64-encoded JPEG images
    class_names: List[str]  # Corresponding detector class names
    track_ids: List[int]  # Track IDs for response mapping

class IdentifyResponse(BaseModel):
    results: List[dict]  # {"track_id": int, "description": str}

def create_prompt(class_names: List[str]) -> str:
    """Create a prompt for the vision model."""
    n = len(class_names)
    if n == 1:
        return (
            f"The object detector classified this as '{class_names[0]}'. "
            "Describe the object in ONE concise sentence (≤ 12 words): "
            "color, shape, key features."
        )
    else:
        numbered = "\n".join(f"{i+1}. <description>" for i in range(n))
        return (
            f"I will show you {n} object crops. The detector classes are: {', '.join(class_names)}.\n"
            "Write ONE short sentence per object (≤ 12 words: color, shape, features).\n"
            f"Reply ONLY in this exact format:\n{numbered}"
        )

def identify_batch(images_b64: List[str], class_names: List[str]) -> List[str]:
    """Run inference on a batch of images."""
    n = len(images_b64)
    
    # Build prompt
    prompt = create_prompt(class_names)
    
    # Build message content
    content = [{"type": "text", "text": prompt}]
    
    for i, img_b64 in enumerate(images_b64):
        if n > 1:
            content.append({
                "type": "text",
                "text": f"Image {i+1} — detected class: {class_names[i]}"
            })
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
        })
    
    # Generate response
    messages = [{"role": "user", "content": content}]
    
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=100 * n,
        temperature=0.2,
        stop=["</s>"]
    )
    
    raw_text = response["choices"][0]["message"]["content"].strip()
    
    # Parse numbered list response
    import re
    results = []
    pattern = re.compile(r"^\s*(\d+)[.)]\s*(.+)$")
    
    if n == 1:
        # Single image: use entire response
        clean = re.sub(r"^1[.)]\s*", "", raw_text).strip()
        results = [clean or class_names[0]]
    else:
        # Multi-image: parse numbered list
        numbered = {}
        for line in raw_text.splitlines():
            m = pattern.match(line)
            if m:
                idx = int(m.group(1)) - 1
                if 0 <= idx < n:
                    numbered[idx] = m.group(2).strip()
        
        for i in range(n):
            if i in numbered:
                results.append(numbered[i])
            else:
                # Fallback to class name
                fallback_lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
                results.append(fallback_lines[i] if i < len(fallback_lines) else class_names[i])
    
    return results

@app.get("/health")
async def health_check():
    """Health check endpoint."""
    return {"status": "healthy", "model": "gemma-3-4b-it", "version": "1.0.0"}

@app.post("/identify", response_model=IdentifyResponse)
async def identify(
    request: IdentifyRequest,
    authorization: Optional[str] = Header(None)
):
    """Identify objects in the provided image crops."""
    # Validate API key if configured
    if SERVER_API_KEY and authorization != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid or missing API key")
    
    # Validate request
    if len(request.images) != len(request.class_names) or len(request.images) != len(request.track_ids):
        raise HTTPException(status_code=400, detail="Mismatched array lengths")
    
    if len(request.images) == 0:
        return IdentifyResponse(results=[])
    
    if len(request.images) > 16:
        raise HTTPException(status_code=400, detail="Batch size too large (max 16)")
    
    try:
        # Run inference
        descriptions = identify_batch(request.images, request.class_names)
        
        # Build response
        results = [
            {"track_id": tid, "description": desc}
            for tid, desc in zip(request.track_ids, descriptions)
        ]
        
        return IdentifyResponse(results=results)
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI app created!")
print(f"   API Key required: {SERVER_API_KEY is not None}")

## Cell 6: Start ngrok Tunnel & Server

Starts the ngrok tunnel and the FastAPI server.

⚠️ **Important**: Keep this notebook running! If the runtime disconnects, the server stops.

### Output to copy:
- `Public URL` — use this in VisionTracker's `--remote-gemma-url` flag
- `API Key` (if set) — use with `--remote-gemma-key`

In [ ]:
import nest_asyncio
import uvicorn

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Create tunnel
public_url = ngrok.connect(8000, "http")
print("━" * 60)
print("🚀 SERVER IS RUNNING!")
print("━" * 60)
print(f"📡 Public URL: {public_url}")
print(f"   Health: {public_url}/health")
print(f"   Identify: {public_url}/identify")
if SERVER_API_KEY:
    print(f"🔑 API Key: {SERVER_API_KEY[:8]}..." + "*" * 10)
else:
    print("🔓 No API key configured (open access)")
print("━" * 60)
print("\n💡 VisionTracker usage:")
print(f"   python main.py --use-remote-gemma --remote-gemma-url {public_url}")
if SERVER_API_KEY:
    print(f"   --remote-gemma-key {SERVER_API_KEY}")
print("\n⚠️  Keep this notebook running! Server will stop if runtime disconnects.")
print("━" * 60)

# Start server
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

---

## 📝 Notes

### Keeping the Server Alive
- Colab free tier will disconnect after ~90 minutes of inactivity
- Keep the browser tab active
- Avoid closing your laptop/sleeping the machine

### ngrok Free Tier Limits
- 1 active tunnel at a time
- ~2 hour session limit (re-run cells to get new URL)
- 40 connections/minute rate limit (more than enough for VisionTracker)

### Troubleshooting

**Out of Memory errors:**
- Reduce batch size in VisionTracker: `--batch-size 2`
- Use CPU-only mode: set `n_gpu_layers=0` in Cell 4

**Slow inference:**
- First request after loading is slow (CUDA warmup)
- Subsequent requests should be 1-3 seconds

**Connection refused:**
- Make sure this notebook is still running
- Check that the ngrok tunnel is active

### Alternative: Cloudflare Tunnel
If ngrok doesn't work, you can use Cloudflare tunnel:
```python
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'])
```